# JS09: Tugas Lab 1

Percobaan metrik jarak pada Annoy, FAISS, dan HNSW untuk 1.000 serta 1.000.000 data, masing masing dalam 2D dan 5D.

## Instalasi library

In [1]:
%pip install -q annoy==1.17.3 faiss-cpu==1.13.2 hnswlib==0.8.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: C:\Users\syahr\AppData\Local\Programs\Python\Python313\python.exe -m pip install --upgrade pip


## Import library

In [2]:
import gc
import time
import numpy as np
import pandas as pd
import faiss
import hnswlib
from annoy import AnnoyIndex

## Pengaturan percobaan

Metrik yang dibandingkan adalah Euclidean dan Cosine. Annoy memakai angular untuk pencarian berdasarkan arah vektor. FAISS memakai inner product pada vektor yang dinormalisasi L2, sedangkan HNSW memakai cosine. Metrik tersebut sejalan dalam mengukur kedekatan arah, tetapi hasil pencarian ANN dapat berbeda dari exact NN.

Setiap percobaan mencari sepuluh tetangga untuk 100 query yang sama. Query dijalankan satu per satu pada semua algoritma. Annoy memakai 10 pohon. HNSW memakai M=16, ef_construction=200, dan ef=50. FAISS memakai indeks Flat, seperti Lab 4.

In [3]:
k = 10
n_queries = 100
sizes = [1_000, 1_000_000]
dimensions = [2, 5]
metrics = ['Euclidean', 'Cosine']
records = []

## Fungsi pengukuran

In [4]:
def run_experiment(X, queries, metric, algorithm):
    dim = X.shape[1]
    if algorithm == 'ANNOY':
        start = time.perf_counter()
        index = AnnoyIndex(dim, 'euclidean' if metric == 'Euclidean' else 'angular')
        index.set_seed(42)
        for i, vector in enumerate(X):
            index.add_item(i, vector)
        index.build(10)
        build_time = time.perf_counter() - start

        start = time.perf_counter()
        neighbors = [index.get_nns_by_vector(query, k) for query in queries]
        query_time = time.perf_counter() - start

    elif algorithm == 'FAISS':
        start = time.perf_counter()
        if metric == 'Euclidean':
            index = faiss.IndexFlatL2(dim)
            database = X
            query_data = queries
        else:
            database = X.copy()
            query_data = queries.copy()
            faiss.normalize_L2(database)
            faiss.normalize_L2(query_data)
            index = faiss.IndexFlatIP(dim)
        index.add(database)
        build_time = time.perf_counter() - start

        start = time.perf_counter()
        neighbors = [index.search(query.reshape(1, -1), k)[1][0] for query in query_data]
        query_time = time.perf_counter() - start

    else:
        start = time.perf_counter()
        index = hnswlib.Index(space='l2' if metric == 'Euclidean' else 'cosine', dim=dim)
        index.init_index(max_elements=len(X), ef_construction=200, M=16, random_seed=42)
        index.add_items(X)
        index.set_ef(50)
        build_time = time.perf_counter() - start

        start = time.perf_counter()
        neighbors = [index.knn_query(query.reshape(1, -1), k=k)[0][0] for query in queries]
        query_time = time.perf_counter() - start

    assert len(neighbors) == len(queries)
    assert all(len(row) == k and len(set(row)) == k for row in neighbors)
    return build_time, query_time * 1000 / len(queries)

## Menjalankan seluruh kombinasi

Data dibuat sekali untuk setiap kombinasi jumlah dan dimensi, lalu dipakai oleh semua metrik dan algoritma. Ukuran satu juta data tetap digunakan.

In [5]:
for n_data in sizes:
    for dim in dimensions:
        rng = np.random.default_rng(42)
        X = rng.random((n_data, dim)).astype(np.float32)
        queries = rng.random((n_queries, dim)).astype(np.float32)
        for metric in metrics:
            for algorithm in ['ANNOY', 'FAISS', 'HNSW']:
                build_time, query_time = run_experiment(X, queries, metric, algorithm)
                records.append({'Jumlah data': n_data, 'Dimensi': dim, 'Metrik': metric,
                                'Algoritma': algorithm, 'Build (detik)': build_time,
                                'Query (ms/query)': query_time})
                print(f'{n_data}/{dim}D, {metric}, {algorithm}: '
                      f'build {build_time:.4f} detik, query {query_time:.6f} ms/query', flush=True)
                gc.collect()
        del X, queries

performance = pd.DataFrame(records)
assert len(performance) == 24

1000/2D, Euclidean, ANNOY: build 0.0135 detik, query 0.020805 ms/query


1000/2D, Euclidean, FAISS: build 0.0001 detik, query 0.015956 ms/query


1000/2D, Euclidean, HNSW: build 0.0239 detik, query 0.020820 ms/query


1000/2D, Cosine, ANNOY: build 0.0126 detik, query 0.018662 ms/query


1000/2D, Cosine, FAISS: build 0.0004 detik, query 0.008606 ms/query


1000/2D, Cosine, HNSW: build 0.0138 detik, query 0.015675 ms/query


... (14 outputs left out)

1000000/5D, Euclidean, HNSW: build 58.8384 detik, query 0.061142 ms/query


1000000/5D, Cosine, ANNOY: build 24.0954 detik, query 0.068354 ms/query


1000000/5D, Cosine, FAISS: build 0.0257 detik, query 7.976587 ms/query


1000000/5D, Cosine, HNSW: build 67.8633 detik, query 0.065047 ms/query


## Tabel performa

Tabel dipisahkan menurut metrik dan jenis waktu agar satuannya jelas. Waktu pembangunan FAISS untuk Cosine mencakup normalisasi L2.

In [6]:
for metric in metrics:
    subset = performance[performance['Metrik'] == metric].copy()
    subset['Jumlah data/Dimensi'] = (subset['Jumlah data'].astype(str) + '/' +
                                    subset['Dimensi'].astype(str) + 'D')
    for value, label in [('Build (detik)', 'Waktu pembangunan (detik)'),
                         ('Query (ms/query)', 'Waktu pencarian (ms/query)')]:
        table = subset.pivot(index='Jumlah data/Dimensi', columns='Algoritma', values=value)
        order = [f'{n}/{d}D' for n in sizes for d in dimensions]
        table = table.reindex(index=order, columns=['ANNOY', 'FAISS', 'HNSW'])
        print(f'\n{metric}: {label}')
        print(table.to_string(float_format=lambda value: f'{value:.6f}'))


Euclidean: Waktu pembangunan (detik)
Algoritma               ANNOY    FAISS      HNSW
Jumlah data/Dimensi                             
1000/2D              0.013507 0.000087  0.023945
1000/5D              0.006955 0.000045  0.024748
1000000/2D          11.496625 0.001698 39.944080
1000000/5D          11.157643 0.004574 58.838427

Euclidean: Waktu pencarian (ms/query)
Algoritma              ANNOY    FAISS     HNSW
Jumlah data/Dimensi                           
1000/2D             0.020805 0.015956 0.020820
1000/5D             0.014886 0.013833 0.020918
1000000/2D          0.046969 3.837528 0.038783
1000000/5D          0.036822 4.154196 0.061142

Cosine: Waktu pembangunan (detik)
Algoritma               ANNOY    FAISS      HNSW
Jumlah data/Dimensi                             
1000/2D              0.012595 0.000358  0.013798
1000/5D              0.008757 0.000076  0.018477
1000000/2D          47.173671 0.009421 58.162336
1000000/5D          24.095398 0.025669 67.863337

Cosine: Waktu pen

## Analisis

In [7]:
for metric in metrics:
    subset = performance[performance['Metrik'] == metric]
    for (n_data, dim), group in subset.groupby(['Jumlah data', 'Dimensi'], sort=False):
        fastest_build = group.loc[group['Build (detik)'].idxmin(), 'Algoritma']
        fastest_query = group.loc[group['Query (ms/query)'].idxmin(), 'Algoritma']
        print(f'{metric}, {n_data}/{dim}D: pembangunan tercepat {fastest_build}, '
              f'pencarian tercepat {fastest_query}.')

Euclidean, 1000/2D: pembangunan tercepat FAISS, pencarian tercepat FAISS.
Euclidean, 1000/5D: pembangunan tercepat FAISS, pencarian tercepat FAISS.
Euclidean, 1000000/2D: pembangunan tercepat FAISS, pencarian tercepat HNSW.
Euclidean, 1000000/5D: pembangunan tercepat FAISS, pencarian tercepat ANNOY.
Cosine, 1000/2D: pembangunan tercepat FAISS, pencarian tercepat FAISS.
Cosine, 1000/5D: pembangunan tercepat FAISS, pencarian tercepat FAISS.
Cosine, 1000000/2D: pembangunan tercepat FAISS, pencarian tercepat ANNOY.
Cosine, 1000000/5D: pembangunan tercepat FAISS, pencarian tercepat HNSW.


IndexFlat FAISS memeriksa seluruh data, sementara Annoy dan HNSW membangun struktur pencarian lebih dahulu. Tabel menunjukkan biaya pembangunan dan pencarian secara terpisah. Hasil ini berlaku untuk perangkat dan pengaturan pada percobaan ini.